In [1]:
!pip install biopython

Defaulting to user installation because normal site-packages is not writeable
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.3/3.3 MB 48.2 MB/s eta 0:00:00


In [3]:
from Bio.PDB import PDBParser, PPBuilder

def parse_chains_and_ligands_with_atom_sequences(pdb_file):
    parser = PDBParser(QUIET=True)
    structure = parser.get_structure('pdb_structure', pdb_file)
    ppb = PPBuilder()

    results = {}

    for model in structure:
        for chain in model:
            chain_id = chain.id

            # Extract protein sequence
            seq = ''
            for pp in ppb.build_peptides(chain):
                seq += str(pp.get_sequence())

            # Extract ligands connected to this chain with their "atom sequence"
            ligands = []
            for residue in chain:
                hetfield, resseq, icode = residue.get_id()
                if hetfield != ' ' and residue.get_resname() != 'HOH':
                    atom_sequence = []
                    for atom in residue:
                        atom_sequence.append({
                            'atom_name': atom.get_name(),
                            'element': atom.element
                        })
                    ligands.append({
                        'residue_name': residue.get_resname(),
                        'resseq': resseq,
                        'insertion_code': icode,
                        'atom_sequence': atom_sequence
                    })

            results[chain_id] = {
                'protein_sequence': seq,
                'ligands': ligands
            }

    return results


# Example usage

!wget -qnc https://files.rcsb.org/download/1HHO.pdb

pdb_file = '/content/1HHO.pdb'  # replace with your file path
data = parse_chains_and_ligands_with_atom_sequences(pdb_file)

# Print results
for chain_id, content in data.items():
    print(f"=== Chain {chain_id} ===")
    print("Protein Sequence:")
    print(content['protein_sequence'])
    print("Ligands:")
    if content['ligands']:
        for lig in content['ligands']:
            print(f"  Ligand {lig['residue_name']} {lig['resseq']}{lig['insertion_code']}:")
            print("    Atom Sequence:")
            atom_seq_str = ' - '.join([f"{a['atom_name']}({a['element']})" for a in lig['atom_sequence']])
            print(f"    {atom_seq_str}")
    else:
        print("  None")
    print()


ModuleNotFoundError: No module named 'Bio'

In [ ]:
from Bio.PDB import PDBParser, NeighborSearch, Selection, PPBuilder

def find_ligand_binding_sites(pdb_file, distance_cutoff=5.0):
    parser = PDBParser(QUIET=True)
    structure = parser.get_structure('pdb_structure', pdb_file)

    model = next(structure.get_models())  # get first model

    # Build list of all protein atoms for NeighborSearch
    atoms = Selection.unfold_entities(model, 'A')  # 'A' for atoms
    ns = NeighborSearch(atoms)

    results = []

    for chain in model:
        chain_id = chain.id

        # Extract protein sequence
        ppb = PPBuilder()
        seq = ''
        for pp in ppb.build_peptides(chain):
            seq += str(pp.get_sequence())

        # Find ligands in this chain
        for residue in chain:
            hetfield, resseq, icode = residue.get_id()
            if hetfield != ' ' and residue.get_resname() != 'HOH':
                ligand_atoms = list(residue.get_atoms())
                ligand_name = residue.get_resname()

                # Find nearby residues (binding site)
                binding_residues = set()
                for atom in ligand_atoms:
                    neighbors = ns.search(atom.get_coord(), distance_cutoff, level='R')
                    for neighbor in neighbors:
                        if neighbor.get_parent().id == chain_id:
                            # Only include protein residues, skip heteroatoms
                            n_hetfield = neighbor.get_id()[0]
                            if n_hetfield == ' ':
                                res_id = neighbor.get_id()[1]
                                res_name = neighbor.get_resname()
                                binding_residues.add( (res_id, res_name) )

                results.append({
                    'chain_id': chain_id,
                    'ligand_name': ligand_name,
                    'ligand_resseq': resseq,
                    'binding_site_residues': sorted(binding_residues)
                })

    return results

# Example usage
pdb_file = '/content/1HHO.pdb'  # replace with your file path
binding_sites = find_ligand_binding_sites(pdb_file)

# Print results
for entry in binding_sites:
    print(f"Chain {entry['chain_id']} - Ligand {entry['ligand_name']} {entry['ligand_resseq']}")
    print("Binding site residues:")
    for res in entry['binding_site_residues']:
        print(f"  Residue {res[1]} {res[0]}")
    print()


In [ ]:
def fetch_json(url):
    r = requests.get(url)
    if r.status_code != 200:
        raise Exception(f"Failed to fetch {url} — {r.status_code}")
    return r.json()

def get_structure_summary_df(pdb_id):
    entry_url = f"https://data.rcsb.org/rest/v1/core/entry/{pdb_id}"
    entry_data = fetch_json(entry_url)

    polymer_ids = entry_data.get("rcsb_entry_container_identifiers", {}).get("polymer_entity_ids", [])
    ligand_ids = entry_data.get("rcsb_entry_container_identifiers", {}).get("non_polymer_entity_ids", [])

    # Full extraction (even if only some is returned)
    molecule_list = []
    chains_list = []
    gene_names_list = []
    organisms_list = []
    lengths_list = []
    mutations_list = []

    for entity_id in polymer_ids:
        poly_url = f"https://data.rcsb.org/rest/v1/core/polymer_entity/{pdb_id}/{entity_id}"
        poly = fetch_json(poly_url)

        molecule = poly.get("entity", {}).get("pdbx_description", "N/A")
        chains = poly.get("rcsb_polymer_entity", {}).get("pdbx_strand_id", [])
        orgs = [o.get("ncbi_scientific_name", "N/A") for o in poly.get("rcsb_entity_source_organism", [])]
        gene_names = [g.get("value") for g in poly.get("rcsb_entity_source_organism", [{}])[0].get("rcsb_gene_name", [])] if poly.get("rcsb_entity_source_organism") else []
        length = poly.get("entity_poly", {}).get("rcsb_sample_sequence_length", "N/A")
        mutations = poly.get("entity_poly", {}).get("rcsb_mutation_count", "N/A")

        molecule_list.append(molecule)
        chains_list.append(", ".join(chains))
        gene_names_list.append(", ".join(gene_names) if gene_names else "N/A")
        organisms_list.append(", ".join(orgs))
        lengths_list.append(length)
        mutations_list.append(mutations)


    ligand_names = []
    for entity_id in ligand_ids:
        lig_url = f"https://data.rcsb.org/rest/v1/core/nonpolymer_entity/{pdb_id}/{entity_id}"
        lig = fetch_json(lig_url)
        nonpoly = lig.get("pdbx_entity_nonpoly", {})
        name = nonpoly.get("name", "N/A")
        comp_id = nonpoly.get("comp_id", "N/A")
        ligand_names.append(f"{comp_id}: {name}")

    return pd.DataFrame([{
        "PDB_ID": pdb_id,
        "Molecules": molecule_list,
        "Chains": chains_list,
        "Gene_Names": gene_names_list,
        "Organisms": organisms_list,
        "Sequence_Lengths": lengths_list,
        "Mutations": mutations_list,
        "Ligands": ligand_names
    }])


In [ ]:
import pandas as pd
csvpath = '/content/merged_with_rmsd.csv'
df2 = pd.read_csv(csvpath)
df2.head()

,pubchem_cid,target_name,malaria_name,target_id,malaria_id,rmsd,alignment_length,cids,protacxns,geneids,pmid,Bit_score,Alignment_Length,Subject_ID,E_value
0,5564,1C14,3AM3,1,A,0.821,208,5892|5564,P0AEK6|P0AEK4,945870|93775413,10595560.0,87.0,237,pdb|3AM3|A,6.740000e-20
1,5564,1C14,1UH5,1,A,0.821,208,5892|5564,P0AEK6|P0AEK4,945870|93775413,10595560.0,85.5,237,pdb|1UH5|A,2.780000e-19
2,5564,1C14,2OL4,1,A,0.788,206,5892|5564,P0AEK6|P0AEK4,945870|93775413,10595560.0,84.3,237,pdb|2OL4|A,7.720000e-19
3,5564,1C14,1VRW,1,A,0.819,208,5892|5564,P0AEK6|P0AEK4,945870|93775413,10595560.0,84.0,237,pdb|1VRW|A,8.040000e-19
4,5564,1C14,3AM5,1,A,0.812,208,5892|5564,P0AEK6|P0AEK4,945870|93775413,10595560.0,83.6,237,pdb|3AM5|A,1.370000e-18


In [ ]:
import pandas as pd
import requests

#FOR THE TARGETS
# Step 1: Create lookup table from df2['target_chain_id']
df_list = []


for pdb in df2['target_name']:
    try:
      #drop chain ID
        pdb = pdb[:4]
        df = get_structure_summary_df(pdb)
        df_list.append(df)
    except Exception as e:
        print(f"Error processing {pdb}: {e}")

# Step 2: Combine all fetched info
combined_df = pd.concat(df_list, ignore_index=True)
lookup = {
    row["PDB_ID"]: {
        "Gene_Names": row["Gene_Names"],
        "Ligands": row["Ligands"]
    }
    for _, row in combined_df.iterrows()
}

# Step 3: Add to original df2
df2["target_gene_names"] = df2["target_name"].str[:4].apply(lambda x: lookup.get(x, {}).get("Gene_Names", ["N/A"]))
df2["target_ligands"] = df2["target_name"].str[:4].apply(lambda x: lookup.get(x, {}).get("Ligands", ["N/A"]))


In [ ]:
df2.head(100)

,pubchem_cid,target_name,malaria_name,target_id,malaria_id,rmsd,alignment_length,cids,protacxns,geneids,pmid,Bit_score,Alignment_Length,Subject_ID,E_value,target_gene_names,target_ligands
0,5564,1C14,3AM3,1,A,0.821,208,5892|5564,P0AEK6|P0AEK4,945870|93775413,10595560.0,87.0,237,pdb|3AM3|A,6.740000e-20,[N/A],"[NAD: NICOTINAMIDE-ADENINE-DINUCLEOTIDE, TCL: ..."
1,5564,1C14,1UH5,1,A,0.821,208,5892|5564,P0AEK6|P0AEK4,945870|93775413,10595560.0,85.5,237,pdb|1UH5|A,2.780000e-19,[N/A],"[NAD: NICOTINAMIDE-ADENINE-DINUCLEOTIDE, TCL: ..."
2,5564,1C14,2OL4,1,A,0.788,206,5892|5564,P0AEK6|P0AEK4,945870|93775413,10595560.0,84.3,237,pdb|2OL4|A,7.720000e-19,[N/A],"[NAD: NICOTINAMIDE-ADENINE-DINUCLEOTIDE, TCL: ..."
3,5564,1C14,1VRW,1,A,0.819,208,5892|5564,P0AEK6|P0AEK4,945870|93775413,10595560.0,84.0,237,pdb|1VRW|A,8.040000e-19,[N/A],"[NAD: NICOTINAMIDE-ADENINE-DINUCLEOTIDE, TCL: ..."
4,5564,1C14,3AM5,1,A,0.812,208,5892|5564,P0AEK6|P0AEK4,945870|93775413,10595560.0,83.6,237,pdb|3AM5|A,1.370000e-18,[N/A],"[NAD: NICOTINAMIDE-ADENINE-DINUCLEOTIDE, TCL: ..."
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
95,5564,2QIO,3AM3,1,A,0.730,195,5892|5564,NaN,NaN,18663709.0,97.4,323,pdb|3AM3|A,9.170000e-24,[fabI],"[NAD: NICOTINAMIDE-ADENINE-DINUCLEOTIDE, TCL: ..."
96,5564,2QIO,1VRW,1,A,0.760,196,5892|5564,NaN,NaN,18663709.0,97.1,323,pdb|1VRW|A,1.410000e-23,[fabI],"[NAD: NICOTINAMIDE-ADENINE-DINUCLEOTIDE, TCL: ..."
97,5564,2QIO,1UH5,1,A,0.734,194,5892|5564,NaN,NaN,18663709.0,97.1,323,pdb|1UH5|A,1.420000e-23,[fabI],"[NAD: NICOTINAMIDE-ADENINE-DINUCLEOTIDE, TCL: ..."
98,5564,2QIO,2OL4,1,A,0.709,194,5892|5564,NaN,NaN,18663709.0,97.1,323,pdb|2OL4|A,1.510000e-23,[fabI],"[NAD: NICOTINAMIDE-ADENINE-DINUCLEOTIDE, TCL: ..."


In [ ]:
#FOR THE MALARIA

# Step 1: Create lookup table from df2['target_chain_id']
df_list = []

for pdb in df2['malaria_name']:
    try:
        pdb = pdb[:4]
        df = get_structure_summary_df(pdb)
        df_list.append(df)
    except Exception as e:
        print(f"Error processing {pdb}: {e}")

# Step 2: Combine all fetched info
combined_df = pd.concat(df_list, ignore_index=True)
lookup = {
    row["PDB_ID"]: {
        "Gene_Names": row["Gene_Names"],
        "Ligands": row["Ligands"]
    }
    for _, row in combined_df.iterrows()
}

# Step 3: Add to original df2
df2["malaria_gene_names"] = df2["malaria_name"].str[:4].apply(lambda x: lookup.get(x, {}).get("Gene_Names", ["N/A"]))
df2["malaria__ligands"] = df2["malaria_name"].str[:4].apply(lambda x: lookup.get(x, {}).get("Ligands", ["N/A"]))


In [ ]:
df2.head(10)

,pubchem_cid,target_name,malaria_name,target_id,malaria_id,rmsd,alignment_length,cids,protacxns,geneids,pmid,Bit_score,Alignment_Length,Subject_ID,E_value,target_gene_names,target_ligands,malaria_gene_names,malaria__ligands
0,5564,1C14,3AM3,1,A,0.821,208,5892|5564,P0AEK6|P0AEK4,945870|93775413,10595560.0,87.0,237,pdb|3AM3|A,6.740000e-20,[N/A],"[NAD: NICOTINAMIDE-ADENINE-DINUCLEOTIDE, TCL: ...",[FabI],"[NAD: NICOTINAMIDE-ADENINE-DINUCLEOTIDE, TCL: ..."
1,5564,1C14,1UH5,1,A,0.821,208,5892|5564,P0AEK6|P0AEK4,945870|93775413,10595560.0,85.5,237,pdb|1UH5|A,2.780000e-19,[N/A],"[NAD: NICOTINAMIDE-ADENINE-DINUCLEOTIDE, TCL: ...",[FabI],"[TCL: TRICLOSAN, NAD: NICOTINAMIDE-ADENINE-DIN..."
2,5564,1C14,2OL4,1,A,0.788,206,5892|5564,P0AEK6|P0AEK4,945870|93775413,10595560.0,84.3,237,pdb|2OL4|A,7.720000e-19,[N/A],"[NAD: NICOTINAMIDE-ADENINE-DINUCLEOTIDE, TCL: ...",[FabI],"[NAD: NICOTINAMIDE-ADENINE-DINUCLEOTIDE, JPN: ..."
3,5564,1C14,1VRW,1,A,0.819,208,5892|5564,P0AEK6|P0AEK4,945870|93775413,10595560.0,84.0,237,pdb|1VRW|A,8.040000e-19,[N/A],"[NAD: NICOTINAMIDE-ADENINE-DINUCLEOTIDE, TCL: ...",[N/A],"[NAI: 1,4-DIHYDRONICOTINAMIDE ADENINE DINUCLEO..."
4,5564,1C14,3AM5,1,A,0.812,208,5892|5564,P0AEK6|P0AEK4,945870|93775413,10595560.0,83.6,237,pdb|3AM5|A,1.370000e-18,[N/A],"[NAD: NICOTINAMIDE-ADENINE-DINUCLEOTIDE, TCL: ...",[fabI],"[NAD: NICOTINAMIDE-ADENINE-DINUCLEOTIDE, TCL: ..."
5,5564,1C14,1NHG,1,A,0.799,154,5892|5564,P0AEK6|P0AEK4,945870|93775413,10595560.0,70.1,141,pdb|1NHG|A,2.720000e-14,[N/A],"[NAD: NICOTINAMIDE-ADENINE-DINUCLEOTIDE, TCL: ...","[N/A, N/A]","[NAD: NICOTINAMIDE-ADENINE-DINUCLEOTIDE, TCL: ..."
6,444810,1CET,1CET,1,A,0.000,305,444810,Q27743|1CET_A,NaN,10187806.0,642.0,316,pdb|1CET|A,0.000000e+00,[N/A],"[CLQ: N4-(7-CHLORO-QUINOLIN-4-YL)-N1,N1-DIETHY...",[N/A],"[CLQ: N4-(7-CHLORO-QUINOLIN-4-YL)-N1,N1-DIETHY..."
7,444810,1CET,1CEQ,1,A,0.167,304,444810,Q27743|1CET_A,NaN,10187806.0,639.0,316,pdb|1CEQ|A,0.000000e+00,[N/A],"[CLQ: N4-(7-CHLORO-QUINOLIN-4-YL)-N1,N1-DIETHY...",[N/A],[]
8,444810,1CET,1T24,1,A,0.393,291,444810,Q27743|1CET_A,NaN,10187806.0,637.0,316,pdb|1T24|A,0.000000e+00,[N/A],"[CLQ: N4-(7-CHLORO-QUINOLIN-4-YL)-N1,N1-DIETHY...",[LDH],"[NAD: NICOTINAMIDE-ADENINE-DINUCLEOTIDE, OXQ: ..."
9,5564,1D7O,3AM3,1,A,0.678,272,5892|5564,P80030,NaN,10610777.0,301.0,321,pdb|3AM3|A,3.620000e-102,[N/A],"[NAD: NICOTINAMIDE-ADENINE-DINUCLEOTIDE, TCL: ...",[FabI],"[NAD: NICOTINAMIDE-ADENINE-DINUCLEOTIDE, TCL: ..."


In [ ]:
df2.to_csv('/content/target_malaria_ligand_gene_added.csv', index = False)

In [ ]:
import pandas as pd
import ast

csvpath = '/content/target_malaria_ligand_gene_added.csv'
df = pd.read_csv(csvpath)

#TARGET LIGANDs
df['target_ligands'] = df['target_ligands'].apply(ast.literal_eval)
# Extract only the ligand codes (before colon) into a new column
df['target_ligand_codes'] = df['target_ligands'].apply(lambda lst: [item.split(':')[0].strip() for item in lst])

#MALARIA LIGANDs
df['malaria__ligands'] = df['malaria__ligands'].apply(ast.literal_eval)
# Extract only the ligand codes (before colon) into a new column
df['malaria_ligand_codes'] = df['malaria__ligands'].apply(lambda lst: [item.split(':')[0].strip() for item in lst])


In [ ]:
import pandas as pd
import string

# Suppose your column is like this
# df = pd.read_csv("your_file.csv")  # if loading from file

# Step 1: Get all unique IDs sorted
unique_ids = sorted(df['target_id'].unique())

# Step 2: Map each unique numeric ID to a letter (A-Z, then AA, AB, etc. if needed)
def number_to_chain(n):
    result = ''
    while n > 0:
        n -= 1
        result = chr(n % 26 + ord('A')) + result
        n //= 26
    return result

id_to_chain = {num: number_to_chain(i+1) for i, num in enumerate(unique_ids)}

# Step 3: Map the target_id column to chain labels
df['chain_id'] = df['target_id'].map(id_to_chain)

# Now df['chain_id'] holds 'A', 'B', 'C'... mappings

#add the names of the pdbs with modified chains M T D
df['target_renamed'] = df['target_name'].astype(str) + '_renamed'
df['malaria_renamed'] = df['malaria_name'].astype(str) + '_renamed'


In [ ]:
df.head(1)

,Unnamed: 0,pubchem_id,target_chain_id,malaria_match_id,percent_identity,alignment_length,evalue,target_gene_names,target_ligands,malaria_match_id_gene_names,malaria_match_id_ligands,In AlphaFill,SMILES,malaria_match_sequence,target_chain,malaria_chain,ligand
0,0,11511120,4I24_A,5DYK_A,27.403846,208,1.420000e-13,"['EGFR, ERBB, ERBB1, HER1']",['1C9: (2E)-N-{4-[(3-chloro-4-fluorophenyl)ami...,['PF3D7_1436600'],"['EDO: 1,2-ETHANEDIOL', 'SO4: SULFATE ION', 'G...",['1C9:0'],COc1cc2ncnc(Nc3ccc(F)c(Cl)c3)c2cc1NC(=O)\C=C\C...,MEEDDNLKKGNERNKKKAIFSNDDFTGEDSLMEDHLELREKLSEDI...,A,A,1C9


In [ ]:
df.to_csv('/content/target_malaria_ligand_gene_added_with_chain_id.csv', index = False)

In [ ]:
import os
import requests
import pandas as pd

# Paths
malaria_dir = '/content/drive/MyDrive/Drug Repurposing Project/malariapdb/'
target_dir = '/content/drive/MyDrive/Drug Repurposing Project/targetpdb/'

# Ensure directories exist
os.makedirs(malaria_dir, exist_ok=True)
os.makedirs(target_dir, exist_ok=True)

# Read CSV
df = pd.read_csv('/content/target_malaria_ligand_gene_added_with_chain_id.csv')

# Function to download a PDB file
def download_pdb(pdb_id, save_dir):
    pdb_id = pdb_id.strip()[:4].upper()
    url = f'https://files.rcsb.org/download/{pdb_id}.pdb'
    save_path = os.path.join(save_dir, f"{pdb_id}.pdb")

    if not os.path.exists(save_path):  # Avoid re-downloading
        try:
            response = requests.get(url)
            response.raise_for_status()
            with open(save_path, 'w') as f:
                f.write(response.text)
            print(f" Downloaded: {pdb_id}")
        except Exception as e:
            print(f" Failed: {pdb_id} — {e}")
    else:
        print(f" Already exists: {pdb_id}")

# Download target PDBs
for target in df['target_name']:
    download_pdb(str(target), target_dir)

# Download malaria PDBs
for malaria in df['malaria_name']:
    download_pdb(str(malaria), malaria_dir)


 Downloaded: 1C14
 Already exists: 1C14
 Already exists: 1C14
 Already exists: 1C14
 Already exists: 1C14
 Already exists: 1C14
 Downloaded: 1CET
 Already exists: 1CET
 Already exists: 1CET
 Downloaded: 1D7O
 Already exists: 1D7O
 Already exists: 1D7O
 Already exists: 1D7O
 Already exists: 1D7O
 Already exists: 1D7O
 Already exists: 1D7O
 Downloaded: 1D8A
 Already exists: 1D8A
 Already exists: 1D8A
 Already exists: 1D8A
 Already exists: 1D8A
 Already exists: 1D8A
 Downloaded: 1J3I
 Already exists: 1J3I
 Already exists: 1J3I
 Downloaded: 1J3K
 Already exists: 1J3K
 Already exists: 1J3K
 Downloaded: 1NHG
 Already exists: 1NHG
 Already exists: 1NHG
 Already exists: 1NHG
 Already exists: 1NHG
 Already exists: 1NHG
 Already exists: 1NHG
 Already exists: 1NHG
 Already exists: 1NHG
 Already exists: 1NHG
 Already exists: 1NHG
 Already exists: 1NHG
 Downloaded: 1P45
 Already exists: 1P45
 Already exists: 1P45
 Already exists: 1P45
 Already exists: 1P45
 Already exists: 1P45
 Downloaded: 1QG6
 A

In [ ]:
import os
import requests
import pandas as pd

# Paths
malaria_dir = '/content/drive/MyDrive/Drug Repurposing Project/malariapdb/'
target_dir = '/content/drive/MyDrive/Drug Repurposing Project/targetpdb/'

# Ensure directories exist
os.makedirs(malaria_dir, exist_ok=True)
os.makedirs(target_dir, exist_ok=True)

# Read CSV
df = pd.read_csv('/content/drive/MyDrive/Drug Repurposing Project/Shared files/Malaria_Mapping_with_SMILES_&_sequences_columns_seperated.csv')

# Function to download a PDB file
def download_pdb(pdb_id, save_dir):
    pdb_id = pdb_id.strip()[:4].upper()
    url = f'https://files.rcsb.org/download/{pdb_id}.pdb'
    save_path = os.path.join(save_dir, f"{pdb_id}.pdb")

    if not os.path.exists(save_path):  # Avoid re-downloading
        try:
            response = requests.get(url)
            response.raise_for_status()
            with open(save_path, 'w') as f:
                f.write(response.text)
            print(f" Downloaded: {pdb_id}")
        except Exception as e:
            print(f" Failed: {pdb_id} — {e}")
    else:
        print(f" Already exists: {pdb_id}")

# Download target PDBs
for target in df['target_chain_id']:
    download_pdb(str(target), target_dir)

# Download malaria PDBs
for malaria in df['malaria_match_id']:
    download_pdb(str(malaria), malaria_dir)


 Downloaded: 4I24
 Already exists: 4I24
 Already exists: 4I24
 Already exists: 4I24
 Already exists: 4I24
 Already exists: 4I24
 Downloaded: 4I23
 Already exists: 4I23
 Already exists: 4I23
 Downloaded: 7DI7
 Already exists: 7DI7
 Already exists: 7DI7
 Already exists: 1CET
 Already exists: 1CET
 Already exists: 1CET
 Downloaded: 3FBV
 Already exists: 3FBV
 Already exists: 3FBV
 Already exists: 3FBV
 Already exists: 3FBV
 Already exists: 3FBV
 Already exists: 3FBV
 Already exists: 3FBV
 Already exists: 3FBV
 Already exists: 3FBV
 Already exists: 3FBV
 Already exists: 3FBV
 Already exists: 3FBV
 Already exists: 3FBV
 Already exists: 3FBV
 Already exists: 3FBV
 Already exists: 3FBV
 Already exists: 3FBV
 Already exists: 3FBV
 Already exists: 3FBV
 Already exists: 3FBV
 Already exists: 3FBV
 Already exists: 3FBV
 Already exists: 3FBV
 Already exists: 3FBV
 Already exists: 3FBV
 Already exists: 3FBV
 Already exists: 3FBV
 Already exists: 3FBV
 Already exists: 3FBV
 Already exists: 3FBV
 Alr

#change name to T M and D

In [ ]:
from Bio.PDB import PDBParser, PDBIO, Select
import os
import pandas as pd


class ParsePDB:
    def __init__(self, pdb_name, chain_id, drug_id):
        self.pdb_name = pdb_name
        self.chain_id = chain_id
        self.drug_id = drug_id
        self.parser = PDBParser(QUIET=True)
        self.structure = self.parser.get_structure("struct", pdb_name)

    def accept_chain(self, chain):
        return chain.id == self.chain_id

    def accept_residue(self, residue):
        hetfield, _, _ = residue.get_id()
        return (hetfield == ' ' or residue.get_resname() == self.drug_id)

    def save_pdb(self, out_file):
        io = PDBIO()
        io.set_structure(self.structure)
        io.save(out_file)

    def keep_target(self, out_file):
        class ChainSelector(Select):
            def accept_chain(inner_self, chain):
                return self.accept_chain(chain)
        io = PDBIO()
        io.set_structure(self.structure)
        io.save(out_file, ChainSelector())

    def keep_drug(self, input_file, out_file):
        struct = self.parser.get_structure("filtered", input_file)
        class ResidueSelector(Select):
            def accept_residue(inner_self, residue):
                return self.accept_residue(residue)
        io = PDBIO()
        io.set_structure(struct)
        io.save(out_file, ResidueSelector())

    def keep_malaria(self, input_file, chain_id, out_file):
        struct = self.parser.get_structure("malaria", input_file)
        for model in struct:
            chains_to_remove = []
            for chain in model:
                if chain.id == chain_id:
                    chain.id = 'M'
                else:
                    chains_to_remove.append(chain)
            for ch in chains_to_remove:
                model.detach_child(ch.id)
        io = PDBIO()
        io.set_structure(struct)
        io.save(out_file)

    def rename_target_ligand(self, input_file, chain_id, out_file):
        struct = self.parser.get_structure("target", input_file)
        for model in struct:
            for chain in model:
                if chain.id == chain_id:
                    chain.id = 'T'
                for residue in chain:
                    hetfield, _, _ = residue.get_id()
                    if hetfield != ' ' and residue.get_resname() == self.drug_id:
                        residue.resname = 'D'
        io = PDBIO()
        io.set_structure(struct)
        io.save(out_file)


In [ ]:

def run_preprocessing_pipeline(target_pdb, malaria_pdb, target_chain_id, malaria_chain_id, drug_resname, input_dir, output_dir):
    target_basename = os.path.splitext(target_pdb)[0]
    malaria_basename = os.path.splitext(malaria_pdb)[0]

    target_pdb_path = os.path.join(input_dir, target_pdb)
    malaria_pdb_path = os.path.join(input_dir, malaria_pdb)

    out_chain_only = os.path.join(output_dir, f"{target_basename}_chain_only.pdb")
    out_ligand = os.path.join(output_dir, f"{target_basename}_with_ligand.pdb")
    out_target = os.path.join(output_dir, f"{target_basename}_renamed.pdb")
    out_malaria = os.path.join(output_dir, f"{malaria_basename}_renamed.pdb")

    print(f"\n Processing: {target_pdb} vs {malaria_pdb}")
    print(" Outputs:", out_chain_only, out_ligand, out_target, out_malaria)

    # Step 1: extract & filter chains
    parser = ParsePDB(target_pdb_path, target_chain_id, drug_resname)
    parser.keep_target(out_chain_only)
    parser.keep_drug(out_chain_only, out_ligand)
    parser.rename_target_ligand(out_ligand, target_chain_id, out_target)
    parser.keep_malaria(malaria_pdb_path, malaria_chain_id, out_malaria)

    # Step 2: align renamed structures
    parser = PDBParser(QUIET=True)
    structure1 = parser.get_structure("target", out_target)
    structure2 = parser.get_structure("malaria", out_malaria)

    seq1, coords1, _ = extract_sequence_and_coords(structure1[0]["T"])
    seq2, coords2, _ = extract_sequence_and_coords(structure2[0]["M"])

    fixed_coords, moving_coords = get_aligned_coords(seq1, coords1, seq2, coords2)
    aligned_fixed, aligned_moving, rmsd = iterative_alignment(fixed_coords, moving_coords)

    print(f" Alignment RMSD: {rmsd:.3f}")

def main():
    base_dir = os.path.dirname(os.path.abspath(__file__))
    input_dir = os.path.join(base_dir, "PDBs")
    output_dir = os.path.join(base_dir, "OutputPDBs")
    os.makedirs(output_dir, exist_ok=True)

    csv_path = os.path.join(base_dir, "pairs.csv")
    df = pd.read_csv(csv_path)

    for idx, row in df.iterrows():
        try:
            run_preprocessing_pipeline(
                target_pdb=row['target_pdb'],
                malaria_pdb=row['malaria_pdb'],
                target_chain_id=row['target_chain'],
                malaria_chain_id=row['malaria_chain'],
                drug_resname=row['ligand'].split(':')[0].strip(),
                input_dir=input_dir,
                output_dir=output_dir
            )
        except Exception as e:
            print(f" Error processing row {idx}: {e}")

if __name__ == "__main__":
    main()


In [ ]:
from Bio.PDB import PDBParser, PDBIO, Select

# Load the mixed PDB file
parser = PDBParser(QUIET=True)
structure = parser.get_structure("mixed", "/content/drive/MyDrive/PracticeCodeFiles/mixed.pdb")

# Check models and chains
for model in structure:
    print(f"Model: {model.id}")
    for chain in model:
        print(f"Chain: {chain.id}")

# Save each chain separately
class ChainSelect(Select):
    def __init__(self, chain_id):
        self.chain_id = chain_id
    def accept_chain(self, chain):
        return chain.id == self.chain_id

io = PDBIO()
for model in structure:
    for chain in model:
        io.set_structure(structure)
        filename = f"{chain.id}_from_mixed.pdb"
        io.save(filename, ChainSelect(chain.id))
        print(f"Saved chain {chain.id} to {filename}")



ligands = []

# Iterate through atoms to find ligands (HETATM, excluding water)
for model in structure:
    for chain in model:
        for residue in chain:
            # Skip water
            if residue.id[0] != " " and residue.get_resname() != "HOH":
                ligands.append({
                    "chain": chain.id,
                    "resname": residue.get_resname(),
                    "resid": residue.id[1],
                })

# Print ligands
for ligand in ligands:
    print(f"Ligand: {ligand['resname']} in chain {ligand['chain']} at residue {ligand['resid']}")

from Bio.PDB import PDBParser, PDBIO, Select

# Load the mixed PDB file
parser = PDBParser(QUIET=True)
structure = parser.get_structure("mixed", "/content/drive/MyDrive/PracticeCodeFiles/mixed_renamed.pdb")

# Check models and chains
for model in structure:
    print(f"Model: {model.id}")
    for chain in model:
        print(f"Chain: {chain.id}")

# Save each chain separately
class ChainSelect(Select):
    def __init__(self, chain_id):
        self.chain_id = chain_id
    def accept_chain(self, chain):
        return chain.id == self.chain_id

io = PDBIO()
for model in structure:
    for chain in model:
        io.set_structure(structure)
        filename = f"{chain.id}_from_mixed.pdb"
        io.save(filename, ChainSelect(chain.id))
        print(f"Saved chain {chain.id} to {filename}")


from Bio.PDB import PDBParser
import numpy as np

parser = PDBParser(QUIET=True)
structure = parser.get_structure("mixed", "mixed.pdb")

# Identify ligand residues
ligands = []
for model in structure:
    for chain in model:
        for residue in chain:
            if residue.id[0] != " " and residue.get_resname() != "HOH":
                ligands.append(residue)

if not ligands:
    print("No ligand found in structure.")
    exit()

# Calculate min distance from each chain to any ligand atom
min_distances = {}
for model in structure:
    for chain in model:
        min_dist = float("inf")
        for residue in chain:
            if residue.id[0] == " ":  # standard residues only
                for atom in residue:
                    for ligand in ligands:
                        for latom in ligand:
                            dist = np.linalg.norm(atom.coord - latom.coord)
                            if dist < min_dist:
                                min_dist = dist
        min_distances[chain.id] = min_dist

# Print results
for chain_id, dist in min_distances.items():
    print(f"Chain {chain_id} minimum distance to ligand: {dist:.2f} Å")

# Determine which is closest
closest_chain = min(min_distances, key=min_distances.get)
print(f"\n✅ Chain {closest_chain} is closest to the ligand.")

from Bio.PDB import PDBParser, PDBIO, Select

parser = PDBParser(QUIET=True)
structure = parser.get_structure("mixed", "mixed.pdb")

# Mapping of chain IDs to original PDB filenames
chain_file_map = {
    "A": "original1.pdb",
    "B": "original2.pdb"
}

# Add REMARK lines by writing them manually before saving structure
# PDBIO does not directly add REMARKs to output, so do it in file writing

io = PDBIO()

class ChainFileSelect(Select):
    def accept_chain(self, chain):
        return True

# Save to temporary file first
temp_filename = "temp_mixed.pdb"
io.set_structure(structure)
io.save(temp_filename, ChainFileSelect())

# Now prepend REMARK lines to the saved file
final_filename = "mixed_renamed.pdb"
with open(final_filename, "w") as final_file:
    # Write REMARKs
    for chain_id, pdb_name in chain_file_map.items():
        final_file.write(f"REMARK Chain {chain_id} from {pdb_name}\n")
    # Append the rest of the PDB contents
    with open(temp_filename, "r") as temp_file:
        final_file.write(temp_file.read())

print(f"Saved renamed file with REMARKs to {final_filename}")


In [ ]:
df.head(1)

,Unnamed: 0,pubchem_id,target_chain_id,malaria_match_id,percent_identity,alignment_length,evalue,target_gene_names,target_ligands,malaria_match_id_gene_names,malaria_match_id_ligands,In AlphaFill,SMILES,malaria_match_sequence,target_chain,malaria_chain,ligand
0,0,11511120,4I24_A,5DYK_A,27.403846,208,1.420000e-13,"['EGFR, ERBB, ERBB1, HER1']",['1C9: (2E)-N-{4-[(3-chloro-4-fluorophenyl)ami...,['PF3D7_1436600'],"['EDO: 1,2-ETHANEDIOL', 'SO4: SULFATE ION', 'G...",['1C9:0'],COc1cc2ncnc(Nc3ccc(F)c(Cl)c3)c2cc1NC(=O)\C=C\C...,MEEDDNLKKGNERNKKKAIFSNDDFTGEDSLMEDHLELREKLSEDI...,A,A,1C9


In [ ]:
headers = df.columns.tolist()
print(headers)

['Unnamed: 0', 'pubchem_id', 'target_chain_id', 'malaria_match_id', 'percent_identity', 'alignment_length', 'evalue', 'target_gene_names', 'target_ligands', 'malaria_match_id_gene_names', 'malaria_match_id_ligands', 'In AlphaFill', 'SMILES', 'malaria_match_sequence', 'target_chain', 'malaria_chain', 'ligand']


In [ ]:
df1 = pd.read_csv('/content/updated_with_rmsd.csv')
df2 = pd.read_csv('/content/Malaria_Mapping_with_SMILES_&_sequences_columns_seperated(2).csv')

In [ ]:
df1.head(1)

,pubchem_id,target_chain_id,malaria_match_id,percent_identity,alignment_length,evalue,target_gene_names,target_ligands,malaria_match_id_gene_names,malaria_match_id_ligands,In AlphaFill,SMILES,malaria_match_sequence,target_chain,malaria_chain,ligand,rmsd,alignment_len
0,11511120,4I24_A,5DYK_A,27.403846,208,1.420000e-13,"['EGFR, ERBB, ERBB1, HER1']",['1C9: (2E)-N-{4-[(3-chloro-4-fluorophenyl)ami...,['PF3D7_1436600'],"['EDO: 1,2-ETHANEDIOL', 'SO4: SULFATE ION', 'G...",['1C9:0'],COc1cc2ncnc(Nc3ccc(F)c(Cl)c3)c2cc1NC(=O)\C=C\C...,MEEDDNLKKGNERNKKKAIFSNDDFTGEDSLMEDHLELREKLSEDI...,A,A,1C9,1.959611,121


In [ ]:
df2.head(1)

,Unnamed: 0,pubchem_id,target_chain_id,malaria_match_id,percent_identity,alignment_length,evalue,target_gene_names,target_ligands,malaria_match_id_gene_names,malaria_match_id_ligands,In AlphaFill,SMILES,malaria_match_sequence,target_chain,malaria_chain,ligand
0,0,11511120,4I24_A,5DYK_A,27.403846,208,1.420000e-13,"['EGFR, ERBB, ERBB1, HER1']",['1C9: (2E)-N-{4-[(3-chloro-4-fluorophenyl)ami...,['PF3D7_1436600'],"['EDO: 1,2-ETHANEDIOL', 'SO4: SULFATE ION', 'G...",['1C9:0'],COc1cc2ncnc(Nc3ccc(F)c(Cl)c3)c2cc1NC(=O)\C=C\C...,MEEDDNLKKGNERNKKKAIFSNDDFTGEDSLMEDHLELREKLSEDI...,A,A,1C9
